# Oscillating-field SABRE: Analytical model for a three-spin system
Before studying this notebook, it is recommended to go through the following:
- sabre_3spin_analytical_static.ipynb

In this notebook, we deal with a three-spin model of the SABRE complex consisting of two hydride protons and a target nucleus in an equatorial ligand. We derive an analytical model that describes the polarisation transfer when the SABRE system is subjected to a sinusoidal polarisation transfer field. We derive a time-independent representation of the Hamiltonian using rotating frame transformations, Floquet theory, and average Hamiltonian theory (AHT). The appearance of periodic level anticrossings (LACs) is demonstrated by plotting energies obtained from the numerical diagonalisation of a Floquet matrix. Finally, we obtain a generalised (LAC) condition for the oscillating polarisation transfer fields and show that the off-diagonal elements connecting the states of interest (effective coupling strengths) can be modified. The calculations are made using SymPy and NumPy, and the plotting is performed using Matplotlib.

In [ ]:
from sympy import *
import numpy as np
import matplotlib.pyplot as plt

### Define an example spin system
We make use of an example spin system in this notebook. The system comprises three spins: two hydride protons and one $^{19}\mathrm{F}$ nucleus from an equatorial 3-fluoropyridine ligand. Chemical shifts for the example system are $\delta_1 = \delta_2 = -23.1\ \mathrm{ppm}$ and $\delta_3 = -124.2\ \mathrm{ppm}$. $J$-couplings are $J_{12} = -7.67\ \mathrm{Hz}$, $J_{13} = -0.05\ \mathrm{Hz}$, and $J_{23} = 2.08\ \mathrm{Hz}$. These values, along with gyromagnetic ratios of both nuclei, are defined below.

In [ ]:
d1_val = -23.1*1e-6
d2_val = -23.1*1e-6
d3_val = -124.2*1e-6

J12_val = -7.67
J13_val = -0.05
J23_val = 2.08

y1_val = 267522188
y2_val = 267522188
y3_val = 251762483

### Split the Hamiltonian into components of spin operators
We start from the static-field three-spin Hamiltonian $\hat{H}$ that was calculated calculated in the previous notebook. We further reduce the state space by only considering the states of interest (either $\ket{S\alpha}$ and $\ket{T_+ \beta}$, or $\ket{S\beta}$ and $\ket{T_- \alpha}$). This reduces the Hamiltonian into $2 \times 2$ blocks, which are decomposed into components of spin operators as follows:
$$
\hat{H} = a_0\hat{E} + a_x\hat{I}_x + a_y\hat{I}_y + a_z\hat{I}_z
$$
In the following block of code, we define the necessary symbols and perform a sanity check: We use the decomposition to calculate back the original $2 \times 2$ Hamiltonians to verify that the decomposition is correct. Note that, after the sanity check, we discard the identity component $a_0\hat{E}$ because it only results to a global phase shift and, thus, does not have an effect to the population transfer.

In [ ]:
a0, ax, ay, az = symbols("a_0, a_x, a_y, a_z", real=True)
J12, J13, J23 = symbols("J_12, J_13, J_23", real=True)
w1, w2, w3 = symbols(r"\omega_1, \omega_2, \omega_3", real=True)

Id = Matrix([
    [1, 0],
    [0, 1]
])

Ix = Rational(1/2) * Matrix([
    [0, 1],
    [1, 0]
])

Iy = Rational(1/2) * Matrix([
    [0, -I],
    [I, 0]
])

Iz = Rational(1/2) * Matrix([
    [1, 0],
    [0, -1]
])

H = a0*Id + ax*Ix +ay*Iy + az*Iz
display(H)

H1 = H.subs([
    (a0, (-pi*(2*J12 + J13 + J23) + (w1 + w2)) / 4),
    (ax, sqrt(2) * pi * (J23 - J13) ),
    (ay, 0),
    (az, (-pi*(4*J12 - J13 - J23) + (-w1 - w2 + 2*w3)) / 2)
])
display(simplify(H1))

H2 = H.subs([
    (a0, (-pi*(2*J12 + J13 + J23) - (w1 + w2)) / 4),
    (ax, - sqrt(2) * pi * (J23 - J13)),
    (ay, 0),
    (az, (-pi*(4*J12 - J13 - J23) - (-w1 - w2 + 2*w3)) / 2)
])
display(simplify(H2))

### Construct the Hamiltonian for the sinusoidal magnetic field
In this notebook, we use a sinusoidal magnetic field:
$$B(t) = B_0 + B_1\cos(\omega_p t)$$
Here, $B_0$ is the average offset, $B_1$ the amplitude, and $\omega_p = 2\pi / T$ the angular frequency of the field corresponding to the period $T$. This means that the resonance frequencies $\omega_i$ in the above Hamiltonian become time-dependent and, consequently, the coefficient $a_z$ becomes time-dependent. We correspondingly split the coefficient $a_z$ into a time-independent and a time-dependent part:
$$
a_z = a_{z0} + a_{z1} \cos(\omega_p t)
$$
The Hamiltonian becomes:
$$
\hat{H} = a_x\hat{I}_x + (a_{z0} + a_{z1}\cos(\omega_p t))\hat{I}_z
$$

In [ ]:
az0, az1 = symbols("a_z0, a_z1", real=True)
wp, t = symbols(r"\omega_p, t", positive=True)

H = ax*Ix + (az0 + az1*cos(wp*t))*Iz
display(H)

### Transform the Hamiltonian into a rotating frame
We compute a unitary transformation operator that transforms the Hamiltonian into a rotating frame defined by the time-dependent term $a_{z1}\cos(\omega_p t)\hat{I}_z$. The unitary operator is:
$$
\hat{U}_1 = e^{ia_{z1}\hat{I}_z \int_0^t \cos(\omega_p t')dt'}
$$
The unitary transformation is then applied to the Hamiltonian by:
$$
\tilde{H} = \hat{U}_1\hat{H}\hat{U}_1^{\dag} + i\frac{d\hat{U}_1}{dt}\hat{U}_1^{\dag}
$$

In [ ]:
tp = symbols(r"t'", positive=True)

U1 = exp(I*az1*Iz*integrate(cos(wp*tp), (tp, 0, t)))
display(U1)

Hr1 = U1 * H * U1.H + I*diff(U1, t)*U1.H
Hr1 = simplify(Hr1)
display(Hr1)

### Perform the Jacobi-Anger expansion
We rewrite the off-diagonal terms using Jacobi-Anger expansion:
$$
e^{iz\sin\theta} = \sum_{n=-\infty}^{\infty} J_n(z)e^{in\theta}
$$
In this case, $z = a_{z1}/\omega_p$ and $\theta = \omega_p t$.

In [ ]:
n = symbols("n", integer=True)

jacobi_anger = Sum(besselj(n, az1/wp) * exp(I*n*wp*t), (n, -oo, oo))
jacobi_anger_conj = Sum(besselj(n, az1/wp) * exp(-I*n*wp*t), (n, -oo, oo))
Hr1 = Matrix(Hr1)
Hr1[0, 1] = Rational(1/2)*ax*jacobi_anger
Hr1[1, 0] = Rational(1/2)*ax*jacobi_anger_conj
display(Hr1)

### Analyse the Hamiltonian using Floquet theory
We interpret the Hamiltonian using Floquet theory, where the time-dependent Hamiltonian is expressed as a time-independent Floquet matrix $\Gamma$ in an infinite-dimensional product space of Hilbert and Fourier space, $\mathcal{H} \otimes \mathcal{F}$:
$$
\Gamma =
\begin{bmatrix}
\ddots & \vdots                     &                  &                             &         \\
\cdots & \tilde{H}^{(0)} - \omega_p & \tilde{H}^{(-1)} & \tilde{H}^{(-2)}            &         \\
       & \tilde{H}^{(1)}            & \tilde{H}^{(0)}  & \tilde{H}^{(-1)}            &         \\
       & \tilde{H}^{(2)}            & \tilde{H}^{(1)}  & \tilde{H}^{(0)} + \omega_p  & \cdots  \\
       &                            &                  & \vdots                      & \ddots
\end{bmatrix}
$$
Here, $\tilde{H}^{(n)}$ are terms from the Fourier expansion of the Hamiltonian:
$$
\tilde{H} = \sum_n \tilde{H}^{(n)}e^{in\omega_p t}
$$

For demonstration purposes, we construct a $5 \times 5$ Floquet matrix.

In [ ]:
H0 = Rational(1/2) * Matrix([
    [az0, ax*besselj(0, az1/wp)],
    [ax*besselj(0, az1/wp), -az0]
])
Hn = Rational(1/2) * Matrix([
    [0, ax*besselj(n, az1/wp)],
    [ax*besselj(-n, az1/wp), 0]
])
Z = Matrix([
    [0, 0],
    [0, 0]
])
Id = Matrix([
    [1, 0],
    [0, 1]
])

floquet = BlockMatrix([
    [H0 - 2*wp*Id,      Hn.subs(n,-1),      Hn.subs(n,-2),  Hn.subs(n,-3),      Hn.subs(n,-4)],
    [Hn.subs(n,1),      H0 - wp * Id,       Hn.subs(n,-1),  Hn.subs(n,-2),      Hn.subs(n,-3)],
    [Hn.subs(n,2),      Hn.subs(n,1),       H0,             Hn.subs(n,-1),      Hn.subs(n,-2)],
    [Hn.subs(n,3),      Hn.subs(n,2),       Hn.subs(n,1),   H0 + wp*Id,         Hn.subs(n,-1)],
    [Hn.subs(n,4),      Hn.subs(n,3),       Hn.subs(n,2),   Hn.subs(n,1),       H0 + 2*wp*Id]
])
floquet = Matrix(floquet)

display(floquet)

### Numerical diagonalisation of the Floquet matrix
In principle, we may analytically construct the infinite-dimensional Floquet matrix and obtain the exact energies (quasienergies) by diagonalising the matrix. However, this is not feasible in practice. Instead, we consider the example 3-fluoropyridine SABRE system for which the Floquet matrix, including terms from $n=-5$ to $n=5$, is constructed, and the quasienergies are obtained by numerical diagonalisation. The quasienergies are calculated with static offset field $B_0$ values ranging from $0\ \mathrm{\mu T}$ to $500\ \mathrm{\mu T}$. The amplitude $B_1$ is set to $100\ \mathrm{\mu T}$.

NOTE: This step takes a few minutes due to the high resolution in $B_0$ axis.

In [ ]:
B0 = symbols("B_0", real=True)
B1 = symbols("B_1", real=True)
y1, y2, y3 = symbols(r"\gamma_1, \gamma_2, \gamma_3", real=True)
d1, d2, d3 = symbols(r"\delta_1, \delta_2, \delta_3", real=True)
J12, J13, J23 = symbols("J_12, J_13, J_23", real=True)

n_max = 5
T_val = 10e-3
B0_vals = np.linspace(0, 500e-6, num=5001)
B1_val = 100e-6

floquet = [[None for _ in range(2*n_max+1)] for _ in range(2*n_max+1)]

for n1 in range(-n_max, n_max+1):
    for n2 in range(-n_max, n_max+1):

        if n1 == n2:
            floquet[n1+n_max][n2+n_max] = H0 + n1*wp*Id

        else:
            floquet[n1+n_max][n2+n_max] = Hn.subs(n, n1-n2)

floquet = Matrix(BlockMatrix(floquet))

floquet = floquet.subs([
    (ax, -sqrt(2)* pi * (J23 - J13)),
    (az0, (-pi*(4*J12 - J13 - J23) - B0*(y1*(1+d1) + y2*(1+d2) - 2*y3*(1+d3))) / 2),
    (az1, (-B1*(y1*(1+d1) + y2*(1+d2) - 2*y3*(1+d3))) / 2)
])

floquet = floquet.subs([
    (J12, J12_val),
    (J13, J13_val),
    (J23, J23_val),
    (y1, y1_val),
    (y2, y2_val),
    (y3, y3_val),
    (d1, d1_val),
    (d2, d2_val),
    (d3, d3_val),
    (wp, 2*np.pi / T_val),
    (B1, B1_val),
    (pi, np.pi)
]).evalf()

Fs = [np.array(floquet.subs(B0, B0_vals[i])).astype(np.float64) for i in range(len(B0_vals))]
Fs = np.array(Fs)

eigvals = np.array([np.linalg.eigvals(F) for F in Fs]).real
eigvals = np.array([np.sort(eigval) for eigval in eigvals])

plt.plot(B0_vals*1e6, eigvals[:,2*n_max+1])
plt.plot(B0_vals*1e6, eigvals[:,2*n_max])
plt.xlabel(r"$B_0$ (µT)")
plt.ylabel("Energy (rad/s)")
plt.show()

np.savetxt("eigvals.txt", eigvals[:, 2*n_max:2*n_max+2])

### Transformation to another rotating frame
Plotting the quasienergies shows an appearance of periodic LACs. These occur when two diagonal elements of the Floquet matrix that represent different states (_e.g._ $\ket{S\alpha}$ and $\ket{T_+\beta}$) become equal. We find that this is true when, for some integer $m$:
$$
a_{z0} + m\omega_p = 0
$$

To come up with a $2 \times 2$ Hamiltonian that embeds this condition, we apply another rotating frame transformation inspired by the Floquet theory. We choose $m$ such that $\left| a_{z0} + m\omega_p \right|$ is minimised and take the interaction frame with respect to $m\omega_p$. The unitary operator of interest is:
$$
\hat{U}_2 = e^{-i\hat{I}_z m \omega_p t}
$$
The transformation is applied again by:
$$
\tilde{\tilde{H}} = \hat{U}_2\tilde{H}\hat{U}_2^{\dag} + i\frac{d\hat{U}_2}{dt}\hat{U}_2^{\dag}
$$
The result is manually simplified, as SymPy does not multiply the exponential term inside the sum.

In [ ]:
m = symbols("m", integer=True)

U2 = exp(-I*Iz*m*wp*t)
display(U2)

Hr2 = U2 * Hr1 * U2.H + I*diff(U2, t)*U2.H
Hr2 = simplify(Hr2)
display(Hr2)

# Manually simplify the result
outside_01 = Hr2[0,1].args[0] * Hr2[0,1].args[1] * Hr2[0,1].args[3]
sum_part_01 = Hr2[0,1].args[2]

outside_10 = Hr2[1,0].args[0] * Hr2[1,0].args[1] * Hr2[1,0].args[3]
sum_part_10 = Hr2[1,0].args[2]

Hr2 = Matrix(Hr2)
Hr2[0,1] = Sum(simplify(outside_01 * sum_part_01.function), *sum_part_01.limits)
Hr2[1,0] = Sum(simplify(outside_10 * sum_part_10.function), *sum_part_10.limits)

display(Hr2)

### Apply the average Hamiltonian theory
We calculate the first-order term of the AHT expansion:
$$
\tilde{\tilde{H}}^{(1)}_{\mathrm{eff}} = \frac{1}{T} \int_{0}^{T} \tilde{\tilde{H}} dt
$$
Here, period of the Hamiltonian $T = 2\pi / \omega_p$. SymPy is incapable of performing the integral for the off-diagonal elements, so we will write them manually (only the term $n=m$ will be nonzero).

In [ ]:
Hr2_eff = 1 / (2*pi/wp) * integrate(Hr2, (t, 0, 2*pi/wp))
display(Hr2_eff)

Hr2_eff = Matrix(Hr2_eff)
Hr2_eff[0,1] = Rational(1/2) * besselj(m, az1/wp)
Hr2_eff[1,0] = Rational(1/2) * besselj(m, 2*az1/wp)
display(Hr2_eff)

### Calculating the LAC condition
We have arrived to a time-independent effective Hamiltonian. There exists a LAC when the diagonal elements are equal. We substitute the coefficient $a_{z0}$ and calculate the $B_0$ for the LAC condition. The condition is calculated for transitions $\ket{S\alpha} \rightarrow \ket{T_+\beta}$ and $\ket{S\beta} \rightarrow \ket{T_-\alpha}$.

In [ ]:
eq_LAC = Hr2[0,0] - Hr2[1,1]

eq_LAC1 = eq_LAC.subs(az0,(-pi*(4*J12 - J13 - J23) + (-w1 - w2 + 2*w3) ) / 2) \
                .subs(w1, -y1*B0*(1+d1)) \
                .subs(w2, -y2*B0*(1+d2)) \
                .subs(w3, -y3*B0*(1+d3))
B0_LAC1 = solve(eq_LAC1, B0)[0]
display(B0_LAC1)

eq_LAC2 = eq_LAC.subs(az0,(-pi*(4*J12 - J13 - J23) - (-w1 - w2 + 2*w3) ) / 2) \
                .subs(w1, -y1*B0*(1+d1)) \
                .subs(w2, -y2*B0*(1+d2)) \
                .subs(w3, -y3*B0*(1+d3))
B0_LAC2 = solve(eq_LAC2, B0)[0]
display(B0_LAC2)

### Example LACs: 3-fluoropyridine system
We calculate values for the LAC condition using the example three-spin system. The period $T$ of the cosine wave is chosen to be $10\ \mathrm{ms}$. The LACs are calculated for cases $m=0$ and $m=1$. It is observed that the case $m=0$ reduces to the static field LAC condition.

In [ ]:
T_val = 10e-3

B0_LAC2_m0 = B0_LAC2.subs([
    (J12, J12_val),
    (J13, J13_val),
    (J23, J23_val),
    (y1, y1_val),
    (y2, y2_val),
    (y3, y3_val),
    (d1, d1_val),
    (d2, d2_val),
    (d3, d3_val),
    (wp, 2*np.pi / T_val),
    (pi, np.pi),
    (m, 0)
])
display(B0_LAC2_m0)

B0_LAC2_m1 = B0_LAC2.subs([
    (J12, J12_val),
    (J13, J13_val),
    (J23, J23_val),
    (y1, y1_val),
    (y2, y2_val),
    (y3, y3_val),
    (d1, d1_val),
    (d2, d2_val),
    (d3, d3_val),
    (wp, 2*np.pi / T_val),
    (pi, np.pi),
    (m, 1)
])
display(B0_LAC2_m1)

### Tuning the coupling strengths by adjusting $B_1$
Once the desired LAC condition has been established, we may tune the coupling strengths by adjusting the amplitude $B_1$. For example, let us maximise the effective coupling in the case of $m=1$. The coupling is modulated by the bessel function $J_1(a_{z1}/\omega_p)$ which reaches its maximum absolute value of $0.58$ when the argument $a_{z1}/\omega_p = \pm 1.84$. We solve for $B_1$ that results in the desired value. It is found that $B_1 = \mp 73.2\ \mathrm{\mu T}$ satisfies the condition (since the amplitude cannot be negative, we neglect the negative answer).

In [ ]:
eq_p = az1/wp - 1.84
eq_m = az1/wp + 1.84

eq_p = eq_p.subs(az1, (-B1 * (y1*(1+d1) + y2*(1+d2) -2*y3*(1+d3)))/2)
eq_m = eq_m.subs(az1, (-B1 * (y1*(1+d1) + y2*(1+d2) -2*y3*(1+d3)))/2)

B1_example_p = solve(eq_p, B1)[0]
B1_example_p = B1_example_p.subs([
    (y1, 267522188),
    (y2, 267522188),
    (y3, 251762483),
    (d1, -23.1*1e-6),
    (d2, -23.1*1e-6),
    (d3, -124.2*1e-6),
    (wp, 2*np.pi / (0.010)),
])

B1_example_m = solve(eq_m, B1)[0]
B1_example_m = B1_example_m.subs([
    (y1, 267522188),
    (y2, 267522188),
    (y3, 251762483),
    (d1, -23.1*1e-6),
    (d2, -23.1*1e-6),
    (d3, -124.2*1e-6),
    (wp, 2*np.pi / (0.010)),
])

display(B1_example_p)
display(B1_example_m)